PRECISION

In [4]:
import pandas as pd


def get_top_30_clusters(file_path: str = "harmonization_output.csv"):
    """Reads the CSV file, groups rows with the same value in the
    'harmonized_name' column, and returns the names of the top 30 largest
    clusters.
    """
    df = pd.read_csv(file_path, usecols=["harmonized_name_final"])    
    top_30 = df["harmonized_name_final"].dropna().value_counts().head(30)    
    return top_30.index.tolist()

def generate_validation_csv(input_path: str = "harmonization_output.csv",
    output_path: str = "validate_precision.csv",
    separator: str = " | "):
    """Generates a 30-row CSV file for manual precision validation.
    Columns:harmonized_name: name of the cluster - clean: distinct original values mapped to this harmonized_name
    - validation: empty column for manual review    """
    
    df = pd.read_csv(input_path, usecols=["harmonized_name_final", "clean"])
    top_30_names = get_top_30_clusters(input_path)
    filtered_df = df[df["harmonized_name_final"].isin(top_30_names)]    
    grouped = (
        filtered_df.dropna(subset=["clean"])
        .groupby(["harmonized_name_final"])["clean"]
        .apply(lambda x: separator.join(x.unique()))
    )
    result_df = grouped.reindex(top_30_names).reset_index()
    result_df["validation"] = ""   
    result_df.to_csv(output_path, index=False, encoding="utf-8")
    
# Step 1: Generate the CSV file to be filled manually
generate_validation_csv("harmonization_output.csv", "validate_precision.csv")

In [6]:
def print_validation_results(
    file_path: str = "validated_precision.csv"):
    """Reads the validated CSV file where 'validation' is 0 (all clean values correct)
    or 1 (contains errors), prints the overall accuracy score, and lists the
    incorrect clusters.    """
    
    df = pd.read_csv(file_path)    
    df["validation"] = pd.to_numeric(df["validation"], errors="coerce")   
    validated_df = df.dropna(subset=["validation"])

    total_clusters = len(validated_df)
    correct_count = (validated_df["validation"] == 0).sum() #correct 0
    wrong_df = validated_df[validated_df["validation"] == 1] #wrong 1

    
    accuracy_pct = ((correct_count / total_clusters * 100) if total_clusters > 0 else 0)
    print(
        f"Correct clusters: {correct_count}/{total_clusters}"
        f" ({accuracy_pct:.1f}%)"
    )
    print(f"Wrong clusters:   {len(wrong_df)}/{total_clusters}")
    print("-" * 50)

    # Print details for every wrong cluster (1)
    if wrong_df.empty:
        print("No wrong clusters found!")
    else:
        print("Incorrect clusters details:\n")
        for _, row in wrong_df.iterrows():
            print(f"Harmonized Name: {row['harmonized_name_final']}")
            print(f"Clean Values:    {row['clean']}")
            print("-" * 50)
            
# Step 2: Print validation results
print_validation_results("validated_precision.csv")

Correct clusters: 25/30 (83.3%)
Wrong clusters:   5/30
--------------------------------------------------
Incorrect clusters details:

Harmonized Name: IONQ
Clean Values:    IONQ | IONQ QUANTUM CANADA | IONIC QUANTUM CANADA
--------------------------------------------------
Harmonized Name: NTT NIPPON TELEGRAPH TELEPHONE
Clean Values:    AT T INTELLECTUAL PROPERTY I AMERICAN TELEPHONE TELEGRAPH COMPANY INTELLECTUAL PROPERTY I | NTT NIPPON TELEGRAPH TELEPHONE | NTT | NTT RESEARCH | NTT DATA GROUP
--------------------------------------------------
Harmonized Name: UNIV SCIENCE TECH CHINA
Clean Values:    NORTH CHINA UNIV TECH | SOUTH CHINA UNIV TECH | UNIV ELECTRONIC SCIENCE TECH CHINA | UNIV SCIENCE TECH CHINA | SOUTH UNIV SCIENCE TECH CHINA | CHINESE SCIENCE TECH UNIV | ELECTRONIC SCIENCE TECH UNIV | SOUTH CHINA SCIENCE UNIV | EAST CHINA UNIV SCIENCE TECH
--------------------------------------------------
Harmonized Name: TENCENT TECH SHENZHEN
Clean Values:    SHENZHEN TENCENT COMPUTER

RECALL

In [5]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def get_30_similar_names(
    file_path: str = "harmonization_output.csv"):
    """Finds the top 30 pairs of distinct 'psn_name' values with the highest word-level TF-IDF cosine similarity that have different 'harmonized_name'
    values.
    """
   
    df = pd.read_csv(
        file_path, usecols=["psn_name", "harmonized_name_final"]
    ).dropna()
    df = df.drop_duplicates(subset=["psn_name"]).reset_index(drop=True)

    names = df["psn_name"].astype(str).tolist()
    clusters = df["harmonized_name_final"].astype(str).tolist()

    # Compute standard word-level TF-IDF and cosine similarity
    tfidf_matrix = TfidfVectorizer().fit_transform(names)
    sim_matrix = cosine_similarity(tfidf_matrix)

    
    i_idx, j_idx = np.triu_indices(len(names), k=1)
    sorted_order = np.argsort(sim_matrix[i_idx, j_idx])[::-1]

    # Collect top 30 pairs belonging to different clusters
    similar_pairs = []
    for idx in sorted_order:
        i, j = i_idx[idx], j_idx[idx]
        if clusters[i] != clusters[j]:
            similar_pairs.append((names[i], names[j]))
            if len(similar_pairs) == 30:
                break

    return similar_pairs

def generate_recall_csv(
    input_path: str = "harmonization_output.csv",
    output_path: str = "validate_recall.csv",
    separator: str = " | ",
) -> None:
    """Calls get_30_similar_names and generates a 30-row CSV file for manual validation.
    Columns:
    - similar_names: the pair of similar psn_names separated by |
    - validation: empty column for manual review
    """
   
    similar_pairs = get_30_similar_names(input_path)   
    formatted_pairs = [
        f"{name_1}{separator}{name_2}" for name_1, name_2 in similar_pairs
    ]
   
    result_df = pd.DataFrame(
        {"similar_names": formatted_pairs, "validation": ""}
    )

    # Save to CSV without the numeric index
    result_df.to_csv(output_path, index=False, encoding="utf-8")

generate_recall_csv("harmonization_output.csv", "validate_recall.csv")

In [7]:
def print_recall_results(file_path: str = "validated_recall.csv"):
    """Reads the validated CSV file where 'validation' is 0 (correctly kept separate)
    or 1 (false negative / should have been merged)"""
  
    df = pd.read_csv(file_path)    
    df["validation"] = pd.to_numeric(df["validation"], errors="coerce")    
    validated_df = df.dropna(subset=["validation"])

    total_pairs = len(validated_df)
    correct_count = (validated_df["validation"] == 0).sum()  # correct = 0
    wrong_df = validated_df[validated_df["validation"] == 1]  # wrong = 1

   
    accuracy_pct = ((correct_count / total_pairs * 100) if total_pairs > 0 else 0)
    print(
        f"Correct pairs: {correct_count}/{total_pairs}"
        f" ({accuracy_pct:.1f}%)"
    )
    print(f"Wrong pairs:   {len(wrong_df)}/{total_pairs}")
    print("-" * 50)

    # Print details for every wrong pair (validation == 1)
    if wrong_df.empty:
        print("No wrong pairs found!")
    else:
        print("Incorrect pairs (missed merges):\n")
        for idx, row in enumerate(wrong_df.itertuples(), 1):
            print(f"{idx}. {row.similar_names}")

print_recall_results("validated_recall.csv")

Correct pairs: 13/30 (43.3%)
Wrong pairs:   17/30
--------------------------------------------------
Incorrect pairs (missed merges):

1. COMMISSARIAT A L 'ENERGIE ATOMIQUE ET AUX ENERGIES ALTERNATIVES | Commissariat à l'Energie Atomique et aux Energies Alternatives
2. COMMISSARIAT A L 'ENERGIE ATOMIQUE ET AUX ENERGIES ALTERNATIVES | COMMISSARIAT À L’ENERGIE ATOMIQUE ET AUX ENERGIES ALTERNATIVES
3. COMMISSARIAT A L ''ENERGIE ATOMIQUE ET AUX ENERGIES ALTERNATIVES | Commissariat à l’Energie Atomique et aux Energies Alternatives
4. COMMISSARIAT A L 'ENERGIE ATOMIQUE ET AUX ENERGIES ALTERNATIVES | Commissariat à l’Energie Atomique et aux Energies Alternatives
5. COMMISSARIAT A L ''ENERGIE ATOMIQUE ET AUX ENERGIES ALTERNATIVES | Commissariat à l'Energie Atomique et aux Energies Alternatives
6. COMMISSARIAT A L ''ENERGIE ATOMIQUE ET AUX ENERGIES ALTERNATIVES | COMMISSARIAT À L’ENERGIE ATOMIQUE ET AUX ENERGIES ALTERNATIVES
7. B. G. NEGEV TECHNOLOGIES AND APPLICATIONS LTD., AT BEN-GURION UNIVE